# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven2"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # Not a plain `git pull` -- if this clone has ANY local changes (e.g. leftover
    # checkpoints/outputs from an earlier run in the same runtime that never got pushed),
    # a pull can fail outright ("local changes would be overwritten") and Colab just
    # prints the error and moves on -- training then silently proceeds on stale code with
    # no visible failure until much later (e.g. a non-fast-forward push at the end).
    # fetch + hard reset guarantees this checkout exactly matches origin/{BRANCH} no
    # matter what state it was left in.
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.7 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 36 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  2%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  5%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [  8%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 11%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 13%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 16%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 19%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [6]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

03:59:41 device: cuda
03:59:41 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
03:59:41 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
03:59:49 epoch 1/20  train_loss=0.23157  val_loss=0.10953  (2.9s)
03:59:49   -> saved best checkpoint (val_loss=0.10953) to steven/outputs/patchtst_checkpoint.pt
03:59:51 epoch 2/20  train_loss=0.16731  val_loss=0.11391  (1.9s)
03:59:53 epoch 3/20  train_loss=0.15137  val_loss=0.09968  (2.1s)
03:59:53   -> saved best checkpoint (val_loss=0.09968) to steven/outputs/patchtst_checkpoint.pt
03:59:55 epoch 4/20  train_loss=0.14287  val_loss=0.09491  (1.9s)
03:59:55   -> saved best checkpoint (val_loss=0.09491) to steven/outputs/patchtst_checkpoint.pt
03:59:57 epoch 5/20  train_loss=0.14188  val_loss=0.09595  (1.9s)
03:59:59 epoch 6/20  train_loss=0.13820  val_loss=0.09251  (1.9s)
03:59:59   -> saved best checkpoint (val_loss=0.09251) to steven/outputs/patchtst_checkpoint.pt
04:00:01 epoch 7/20  train_

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [7]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

04:00:29 device: cuda
04:00:30 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
04:00:30 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
04:00:30 price_scale (open_ret, body_ret, upper_wick, lower_wick): [0.002682909369468689, 0.0028937843162566423, 0.0016679799882695079, 0.0017456382047384977]
04:00:33 epoch 1/30  beta=0.20  train_recon=385.83789 (kl=7.8494)  val_recon=4.96551 (kl=1.6561)  (2.5s)
04:00:33   -> saved best checkpoint (val_recon=4.96551) to steven/outputs/cvae_checkpoint.pt
04:00:35 epoch 2/30  beta=0.40  train_recon=6.50629 (kl=1.2610)  val_recon=2.62205 (kl=1.2000)  (1.4s)
04:00:35   -> saved best checkpoint (val_recon=2.62205) to steven/outputs/cvae_checkpoint.pt
04:00:36 epoch 3/30  beta=0.60  train_recon=3.05062 (kl=1.2039)  val_recon=2.34258 (kl=1.2004)  (1.4s)
04:00:36   -> saved best checkpoint (val_recon=2.34258) to steven/outputs/cvae_checkpoint.pt
04:00:38 epoch 4/30  beta=0.80  train_recon=2.60479 (kl=1.

## Evaluate both models on the fixed test set

In [8]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

04:01:19 device: cuda
04:01:19 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
04:01:19 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
04:01:19 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
04:01:19 running walk-forward backtest (ctx=70 bars, patchtst_min_return>=0.100%, cvae_min_return>=0.020% [no CVAE quality gate -- see backlog.md], stop_loss=2.00% [shared, see backlog.md], 24537..27006)...
04:01:25 walk-forward: PatchTST 460 trades / 1477 decisions, CVAE 799 trades / 799 decisions
04:01:25 wrote metrics to steven/outputs/metrics.json
04:01:25 walk_forward: {
  "ctx_bars": 70,
  "patchtst_min_return_threshold": 0.001,
  "cvae_min_return_threshold": 0.0002,
  "stop_loss_pct": 0.02,
  "buy_and_hold": {
    "entry_date": "2024-01-16",
    "entry_price": 474.95,
    "exit_date": "2025-05-30",
    "exit_price": 589.46,
    "elapsed_years": 1.3689253935660506,
    "tota

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [9]:
!python steven/src/update_report.py

04:01:28 updated steven/v1.md: results-samples, hit-summary, spread-summary, buy-hold-benchmark, walk-forward-strategies, walk-forward-outcome-breakdown
04:01:28 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveat in Results, and the 'Retrain both models' checkbox under Next steps.


## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven2` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [ ]:
# %%bash
# git fetch origin steven2
# git merge origin/steven2 --no-edit

In [13]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [ ]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
# Pushes to steven2, not steven -- this notebook and this branch are the working copy for
# now; steven is left alone so a second person's in-flight work there can't collide with
# this runtime's pushes.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven2

### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [15]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/metrics.json (deflated 68%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 8%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 77%)
  adding: steven/outputs/sample_plots/lose_expiry_start25293_ctx70.png (deflated 10%)
  adding: steven/outputs/sample_plots/win_expiry_start25926_ctx70.png (deflated 10%)
  adding: steven/outputs/sample_plots/win_take_profit_start25839_ctx70.png (deflated 11%)
  adding: steven/outputs/sample_plots/lose_stop_loss_start26733_ctx70.png (deflated 10%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>